In [ ]:
#| default_exp lc_node

In [ ]:
#| hide
from nbdev.showdoc import *

# Lifecycle Node

> A managed lifecycle camera node for hardware that needs careful initialization and cleanup.

## Overview

Standard ROS2 nodes start publishing as soon as they are launched. This is fine for software-only nodes, but camera hardware often needs explicit init/cleanup sequences — open device, configure, start streaming, stop streaming, close device.

ROS2 **managed lifecycle** nodes expose a state machine that external tools (or launch files) can control:

```
Unconfigured → Inactive → Active → Finalized
                  ↑          |
                  └──────────┘  (deactivate)
```

This gives you:

- **Controlled startup** — configure hardware before it starts publishing
- **Pause/resume** — deactivate without killing the process
- **Clean shutdown** — guaranteed cleanup of hardware resources

## Imports

In [ ]:
#| export
#| hide
import sys
import rclpy
from rclpy.lifecycle import LifecycleNode, TransitionCallbackReturn
from rclpy.executors import ExternalShutdownException
import cv2
from cv_bridge import CvBridge
from sensor_msgs.msg import Image

## Import Camera from 02

We reuse the `Camera` wrapper from notebook 02.

In [ ]:
#| export
#| echo: false
from ros2_cam_nbdev.cam_pub import Camera

## LcNode

`LcNode` is a `LifecycleNode` that manages camera resources through the lifecycle state machine.

| State | What happens |
|-------|-------------|
| **Unconfigured** | Node is created but camera is not opened |
| **Inactive** (`on_configure`) | Camera is opened, publisher is created, but timer is not running |
| **Active** (`on_activate`) | Timer starts — frames are captured and published |
| **Deactivated** (`on_deactivate`) | Timer stops — no frames are published, camera stays open |
| **Finalized** (`on_cleanup`) | Camera is released, resources cleaned up |

In [ ]:
#| export
class LcNode(LifecycleNode):
    """Managed lifecycle camera node."""
    def __init__(self):
        super().__init__('lc_node')

        # Declare parameters (same as CamPublisher)
        self.declare_parameter('video_source', '0')
        self.declare_parameter('frame_rate', 30.0)
        self.declare_parameter('topic_name', '/camera/image_raw')
        self.declare_parameter('video_fourcc', 'MJPG')
        self.declare_parameter('resolution_width', 640)
        self.declare_parameter('resolution_height', 480)

        # These are set during on_configure
        self.camera = None
        self.publisher_ = None
        self.timer = None
        self.bridge = CvBridge()
        self.frame_count = 0

    def on_configure(self, state):
        """Open the camera and create the publisher. Transition: Unconfigured → Inactive."""
        self.get_logger().info('Configuring...')

        try:
            source = self.get_parameter('video_source').value
            fourcc = self.get_parameter('video_fourcc').value
            width = self.get_parameter('resolution_width').value
            height = self.get_parameter('resolution_height').value
            self.camera = Camera(source, fourcc, width, height)

            topic = self.get_parameter('topic_name').value
            self.publisher_ = self.create_publisher(Image, topic, 20)

            self.get_logger().info(f'Configured — source={source}, topic="{topic}"')
            return TransitionCallbackReturn.SUCCESS
        except Exception as e:
            self.get_logger().error(f'Configuration failed: {e}')
            return TransitionCallbackReturn.FAILURE

    def on_activate(self, state):
        """Start the timer to publish frames. Transition: Inactive → Active."""
        self.get_logger().info('Activating...')

        fps = self.get_parameter('frame_rate').value
        period = 1.0 / fps
        self.timer = self.create_timer(period, self._publish_frame)
        self.frame_count = 0

        self.get_logger().info(f'Active — publishing at {fps} FPS')
        return TransitionCallbackReturn.SUCCESS

    def on_deactivate(self, state):
        """Stop the timer. Transition: Active → Inactive."""
        self.get_logger().info('Deactivating...')

        if self.timer is not None:
            self.timer.cancel()
            self.timer = None

        self.get_logger().info('Inactive — camera still open, not publishing')
        return TransitionCallbackReturn.SUCCESS

    def on_cleanup(self, state):
        """Release the camera. Transition: Inactive → Unconfigured."""
        self.get_logger().info('Cleaning up...')

        if self.camera is not None:
            self.camera.release()
            self.camera = None

        self.publisher_ = None
        self.timer = None

        self.get_logger().info('Cleaned up — camera released')
        return TransitionCallbackReturn.SUCCESS

    def on_shutdown(self, state):
        """Handle shutdown from any state."""
        self.get_logger().info('Shutting down...')
        if self.camera is not None:
            self.camera.release()
        return TransitionCallbackReturn.SUCCESS

    def _publish_frame(self):
        """Capture a frame and publish it."""
        if self.camera is None:
            return

        ret, frame = self.camera.read()
        if not ret:
            self.get_logger().warn('Failed to read frame')
            return

        msg = self.bridge.cv2_to_imgmsg(frame, encoding='bgr8')
        self.publisher_.publish(msg)

        if self.frame_count % 10 == 0:
            self.get_logger().info(f'Published frame {self.frame_count}')
        self.frame_count += 1

### show_doc

In [ ]:
show_doc(LcNode)

---

[source](https://github.com/Ramon-PR/ros2_cam_nbdev/blob/main/ros2_cam_nbdev/lc_node.py#L16){target="_blank" style="float:right; font-size:smaller"}

### LcNode

```python

def LcNode(
    
):


```

*Managed lifecycle camera node.*

## Entry Point

In [ ]:
#| export
def main(args=None):
    """Initialize ROS2, spin the LcNode, and handle shutdown."""
    rclpy.init(args=args)
    node = LcNode()

    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        node.get_logger().info('Node stopped by user or external shutdown')
    finally:
        node.destroy_node()
        rclpy.try_shutdown()

In [ ]:
#| exporti
if not hasattr(sys, 'ps1'):
    if __name__ == "__main__":
        main()

## Usage

First, export the notebook to generate the Python module:

``` bash
uv run nbdev-export
```

This creates `ros2_cam_nbdev/lc_node.py`.

### Method 1 — Run directly with Python

The fastest way to test. No workspace or compilation needed.

**Terminal 1** — run the node:

``` bash
uv run python -m ros2_cam_nbdev.lc_node
```

The node starts in `Unconfigured` state — it does nothing until you transition it.

### Configure and activate

``` bash
ros2 lifecycle set /lc_node configure
```

``` bash
ros2 lifecycle set /lc_node activate
```

Now it should be publishing frames.

**Terminal 2** — check the topic is publishing:

``` bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

### Deactivate and cleanup

``` bash
ros2 lifecycle set /lc_node deactivate
```

``` bash
ros2 lifecycle set /lc_node cleanup
```

### Check state

``` bash
ros2 lifecycle get /lc_node
```

### Method 2 — Full ROS2 workflow

The standard ROS2 way: create a workspace, package, compile, and run.

**Step 1** — Source ROS2:

``` bash
source /opt/ros/$ROS_DISTRO/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 2** — Create the workspace:

``` bash
zsh scripts/generate_ros2_ws.zsh new_ros2_ws
```

**Step 3** — Create the package:

``` bash
zsh scripts/mk_ros2_pkg.sh new_ros2_ws my_camera_pkg camera_opencv.txt
```

> This creates `new_ros2_ws/src/my_camera_pkg/` with ROS2 dependencies
> read from `scripts/pkg_dependencies/camera_opencv.txt`.

**Step 4** — Copy the node into the package:

``` bash
cp ros2_cam_nbdev/cam_pub.py ros2_cam_nbdev/lc_node.py new_ros2_ws/src/my_camera_pkg/my_camera_pkg/
```

**Step 5** — Compile:

``` bash
bash scripts/compile_pkg.sh new_ros2_ws my_camera_pkg
```

> `compile_pkg.sh` generates `setup.py` automatically (entry points are
> discovered from any file with `def main(args=None):`) and runs `colcon build`.

**Step 6** — Source the workspace overlay:

``` bash
source new_ros2_ws/install/setup.bash
```

> Use `setup.zsh` if your shell is zsh.

**Step 7** — Run the node:

``` bash
ros2 run my_camera_pkg lc_node
```

Then configure and activate as shown above in Method 1.

**Terminal 2** — verify and visualize:

``` bash
ros2 topic hz /camera/image_raw
```

See `08_visualization` for tools to view the images.

## Visualize

With the node activated, open a second terminal and visualize the images:

``` sh
ros2 run rqt_image_view rqt_image_view
```

Select `/camera/image_raw` from the dropdown.

For more visualization options (rviz2, Foxglove), see `08_visualization`.

## Next Steps

Now that we have a solid camera publisher pattern (basic, configurable, lifecycle), we can explore transport options in `05_image_transport` and camera calibration in `06_calibration_and_info`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()